# 3. 信用卡欺诈交易检测

## 一、实验简介

本实验围绕 **金融反欺诈（Fraud Detection）** 场景展开，基于 Kaggle Credit Card Fraud Detection 数据集，使用支持向量机（SVM）、随机森林（Random Forest）、XGBoost 以及异常检测方法（Isolation Forest）对信用卡交易进行欺诈识别，并重点分析在**高度不平衡数据**下的模型评估与阈值选择策略。

**为什么反欺诈是一个特殊的机器学习问题？**

1. **极端不平衡**：数据集包含 284,807 笔交易，其中仅 492 笔（约 0.173%）为欺诈。传统的准确率指标在此场景下几乎无意义——一个"全部预测为正常"的模型也能达到 99.83% 的准确率。
2. **非对称代价**：漏掉一笔欺诈交易（假阴性）的损失远远大于误报一笔正常交易（假阳性）的代价。因此 **Recall（召回率）** 是核心指标。
3. **特征已脱敏**：出于隐私保护，原始特征已通过 PCA 变换为 V1–V28，仅保留 `Time` 和 `Amount` 为原始特征。这使得业务解释性受限，但非常适合用来对比模型的纯粹判别能力。

本实验使用 Kaggle 上的 Credit Card Fraud Detection [<sup>1</sup>](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud) 数据集。

## 二、实验目标

- 理解**高度不平衡数据集**对模型训练和评估带来的挑战。
- 掌握不平衡样本处理技术：**欠采样（Under-sampling）、过采样（SMOTE）、类别权重调整（class_weight）**。
- 理解 SVM 的核心思想——**最大间隔超平面（Maximum Margin Hyperplane）** 与 **核技巧（Kernel Trick）**。
- 比较 SVM、随机森林、XGBoost 和 Isolation Forest 在反欺诈任务上的表现差异。
- 深入理解 **PR 曲线（Precision-Recall Curve）** 和 **PR-AUC** 为什么比 ROC-AUC 更适合不平衡场景。
- 学会根据业务需求进行 **分类阈值调优**，在 Precision 与 Recall 之间做出合理取舍。

## 三、实验要求

- 下载并加载 Kaggle Credit Card Fraud Detection 数据集。
- 对数据进行探索性分析，理解类别分布和特征特性。
- 实施至少一种不平衡样本处理方法（SMOTE 或欠采样）。
- 分别构建 SVM（线性核）、随机森林、XGBoost 和 Isolation Forest 模型。
- 绘制 PR 曲线和 ROC 曲线，重点比较 PR-AUC 和 Recall。
- 进行阈值分析，给出反欺诈场景下的最优阈值建议。

## 四、思考提示

1. 在正类仅占 0.17% 的数据上，为什么 Accuracy 不是一个好的评估指标？
2. SMOTE 过采样和随机欠采样各有什么优缺点？在什么情况下选择哪种方法？
3. SVM 在大规模数据上的计算瓶颈是什么？本实验中如何应对？
4. Isolation Forest 作为无监督异常检测方法，与有监督方法相比有什么独特优势和劣势？
5. 在反欺诈场景中，如果要求 Recall ≥ 0.95，Precision 会受到怎样的影响？这在业务上意味着什么？

---

## 1. 环境准备

In [ ]:
from __future__ import annotations

import os
import shutil
import warnings

import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import xgboost as xgb
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from matplotlib import font_manager
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, average_precision_score,
    precision_recall_curve, confusion_matrix, classification_report,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

warnings.filterwarnings("ignore")

接下来配置中文字体，确保图表中的中文标签能正确显示：

In [ ]:
def configure_plot_fonts() -> None:
    """配置 matplotlib 中文字体，避免 seaborn 主题覆盖字体设置。"""
    sns.set_theme(style="whitegrid")
    font_candidates = [
        "Microsoft YaHei", "SimHei", "Noto Sans CJK SC",
        "PingFang SC", "WenQuanYi Zen Hei",
    ]
    available_fonts = {f.name for f in font_manager.fontManager.ttflist}
    selected_font = next(
        (name for name in font_candidates if name in available_fonts), None
    )
    if selected_font is None:
        warnings.warn(
            "未检测到常见中文字体，图表中文可能显示为方块。"
            "建议安装 'Microsoft YaHei' 或 'Noto Sans CJK SC'。",
            UserWarning,
        )
        plt.rcParams["font.sans-serif"] = ["DejaVu Sans"]
    else:
        plt.rcParams["font.sans-serif"] = [selected_font, "DejaVu Sans"]
    plt.rcParams["axes.unicode_minus"] = False


configure_plot_fonts()

---

## 2. 数据集加载与探索性分析

### 2.1 加载数据

请先从 Kaggle 下载数据集 CSV 文件并放置在当前工作目录下。下载地址：https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud

In [ ]:
LOCAL_CSV = "data/creditcard.csv"
if not os.path.exists(LOCAL_CSV):
    print("本地未找到数据集，正在从 Kaggle 下载 ...")
    dataset_path = kagglehub.dataset_download("mlg-ulb/creditcardfraud")
    # kagglehub 返回的是目录路径，CSV 文件在其中
    src = os.path.join(dataset_path, "creditcard.csv")
    # 复制到当前工作目录方便后续使用
    shutil.copy(src, LOCAL_CSV)
    print(f"数据集已保存至：{LOCAL_CSV}")
else:
    print(f"已检测到本地数据集：{LOCAL_CSV}，跳过下载。")
df = pd.read_csv(LOCAL_CSV)

print(f"数据集维度：{df.shape}")
print(f"目标变量分布：\n{df['Class'].value_counts()}")
print(f"欺诈比例：{df['Class'].mean():.5f} ({df['Class'].sum()} / {len(df)})")

### 2.2 类别分布可视化

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

# 类别计数
counts = df["Class"].value_counts()
ax.bar(["正常 (0)", "欺诈 (1)"], counts.values, color=["steelblue", "crimson"])
ax.set_title("交易类别分布")
ax.set_ylabel("交易笔数")
for i, v in enumerate(counts.values):
    ax.text(i, v + 2000, str(v), ha="center", fontweight="bold")

plt.tight_layout()
plt.show()

> **观察**：正常交易与欺诈交易的数量比约为 **578:1**，这是一个典型的极端不平衡分类问题。

### 2.3 特征分布探索

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Amount 分布
for label, color, name in [(0, "steelblue", "正常"), (1, "crimson", "欺诈")]:
    subset = df[df["Class"] == label]["Amount"]
    axes[0].hist(subset, bins=50, alpha=0.6, color=color, label=name, density=True)
axes[0].set_title("交易金额（Amount）分布")
axes[0].set_xlabel("Amount")
axes[0].set_ylabel("密度")
axes[0].set_xlim(0, 1000)  # 截断长尾便于观察
axes[0].legend()

# Time 分布
for label, color, name in [(0, "steelblue", "正常"), (1, "crimson", "欺诈")]:
    subset = df[df["Class"] == label]["Time"]
    axes[1].hist(subset, bins=50, alpha=0.6, color=color, label=name, density=True)
axes[1].set_title("交易时间（Time）分布")
axes[1].set_xlabel("Time (秒)")
axes[1].set_ylabel("密度")
axes[1].legend()

plt.tight_layout()
plt.show()

### 2.4 数据预处理

V1–V28 已经过 PCA 变换，尺度较为一致。但 `Time` 和 `Amount` 的尺度差异较大，需要进行标准化。**SVM 对特征尺度非常敏感**，因此标准化是必要步骤。

In [ ]:
scaler = StandardScaler()
df["Amount_scaled"] = scaler.fit_transform(df[["Amount"]])
df["Time_scaled"] = scaler.fit_transform(df[["Time"]])

# 移除原始 Time 和 Amount 列
df_processed = df.drop(columns=["Time", "Amount"])

X = df_processed.drop(columns=["Class"])
y = df_processed["Class"]

print(f"特征矩阵维度：{X.shape}")
print(f"特征列表：{X.columns.tolist()}")

### 2.5 划分训练集与测试集

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y,
)

print(f"训练集大小：{X_train.shape[0]}，欺诈比例：{y_train.mean():.5f}")
print(f"测试集大小：{X_test.shape[0]}，欺诈比例：{y_test.mean():.5f}")

---

## 3. 核心概念

### 3.1 不平衡数据处理

在极端不平衡数据上直接训练模型，模型会倾向于"偷懒"——将所有样本预测为多数类（正常），从而获得很高的准确率但完全无法识别欺诈。常见处理策略包括：

| 方法 | 原理 | 优点 | 缺点 |
|------|------|------|------|
| **随机欠采样** | 从多数类中随机抽取与少数类相同数量的样本 | 简单快速，减少训练数据量 | 丢失大量多数类信息 |
| **SMOTE 过采样** | 在少数类样本之间插值生成新的合成样本 | 不丢失信息，增加少数类多样性 | 可能引入噪声，训练数据量增大 |
| **类别权重调整** | 在损失函数中给少数类更高的权重 | 不改变数据分布，实现简单 | 权重设置需要调优 |
| **混合策略** | 先 SMOTE 过采样再欠采样 | 平衡数据量与信息保留 | 需要更多调参 |

### 3.2 支持向量机（SVM）

SVM 的核心思想是寻找一个**最大间隔超平面（Maximum Margin Hyperplane）**，使得两个类别之间的"间隔"最大。

**关键概念**：
- **支持向量（Support Vectors）**：离超平面最近的样本点，它们决定了超平面的位置。
- **核技巧（Kernel Trick）**：通过核函数将数据映射到高维空间，使得线性不可分的数据在高维空间中线性可分。常用核函数有线性核（Linear）、RBF 核（高斯核）、多项式核等。
- **软间隔（Soft Margin）**：通过参数 C 控制对误分类的容忍程度。C 越大，对误分类惩罚越重，间隔越小。

**SVM 在本实验中的挑战**：标准 SVM（`sklearn.svm.SVC`）的时间复杂度为 O(n²) ~ O(n³)，在 28 万条数据上训练非常缓慢。因此本实验使用 **`LinearSVC`**（线性核 SVM），其基于 liblinear 实现，时间复杂度为 O(n)，能高效处理大规模数据。

### 3.3 Isolation Forest（孤立森林）

Isolation Forest 是一种**无监督异常检测**算法。其核心思想非常直观：**异常点因为"与众不同"，所以更容易被随机分割孤立出来。**

- 构建方式：随机选择特征、随机选择分裂值，递归分割数据。
- 判别依据：异常点在随机树中的**平均路径长度**更短（更早被孤立）。
- 优势：不需要标签数据，天然适合欺诈检测等"正样本极少"的场景。
- 劣势：无法利用标签信息优化决策边界，通常精度不如有监督方法。

### 3.4 为什么 PR-AUC 比 ROC-AUC 更适合不平衡场景？

**ROC 曲线**绘制的是 TPR（Recall）vs FPR。在极端不平衡数据中，由于负样本（正常交易）数量巨大，即使模型产生大量假正例，FPR 仍然很低（因为分母巨大）。这导致 ROC-AUC 值虚高，给人"模型很好"的错觉。

**PR 曲线**绘制的是 Precision vs Recall。Precision 的分母是"所有被预测为正的样本"，不受负样本数量的影响，因此在不平衡场景下能更真实地反映模型性能。

| 指标 | 适用场景 | 不平衡数据下的表现 |
|------|----------|-------------------|
| **ROC-AUC** | 正负样本较均衡 | 容易虚高，过于乐观 |
| **PR-AUC (Average Precision)** | 正样本极少（如欺诈检测） | 更敏感、更真实 |

---

## 4. 不平衡样本处理

### 4.1 SMOTE 过采样

**注意**：SMOTE 只对**训练集**进行过采样，测试集必须保持原始分布不变，以确保评估结果反映真实场景。

In [ ]:
# Ref: https://imbalanced-learn.org/stable/references/generated/imblearn.over_sampling.SMOTE.html
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print(f"SMOTE 前训练集大小：{X_train.shape[0]}，欺诈数量：{y_train.sum()}")
print(f"SMOTE 后训练集大小：{X_train_smote.shape[0]}，欺诈数量：{y_train_smote.sum()}")
print(f"SMOTE 后类别分布：\n{pd.Series(y_train_smote).value_counts()}")

### 4.2 随机欠采样

In [ ]:
# Ref: https://imbalanced-learn.org/stable/references/generated/imblearn.under_sampling.RandomUnderSampler.html
rus = RandomUnderSampler(random_state=42)
X_train_under, y_train_under = rus.fit_resample(X_train, y_train)

print(f"欠采样后训练集大小：{X_train_under.shape[0]}，欺诈数量：{y_train_under.sum()}")
print(f"欠采样后类别分布：\n{pd.Series(y_train_under).value_counts()}")

> **关键决策**：后续有监督模型训练将使用 **SMOTE 过采样后的训练集** `(X_train_smote, y_train_smote)`，因为欠采样丢失了大量正常交易信息。同时，我们也会通过 `class_weight` 参数作为对比方案。

---

## 5. 构建 SVM 模型

### 5.1 训练线性 SVM

由于数据量较大（SMOTE 后约 45 万条），我们使用 `LinearSVC` 而非 `SVC(kernel='rbf')`。`LinearSVC` 不直接输出概率，因此需要用 `CalibratedClassifierCV` 进行概率校准，以便计算 PR-AUC 等指标。

In [ ]:
# Ref: https://scikit-learn.org/stable/modules/generated/sklearn.svm.LinearSVC.html
# Ref: https://scikit-learn.org/stable/modules/generated/sklearn.calibration.CalibratedClassifierCV.html

# LinearSVC + 概率校准
# 使用 SMOTE 过采样后的数据训练
linear_svc = LinearSVC(
    C=1.0,              # 正则化参数：C 越大对误分类惩罚越重
    max_iter=2000,      # 最大迭代次数
    random_state=42,
    dual="auto",
)

# 通过 CalibratedClassifierCV 包装以获得概率输出
# method="sigmoid" 使用 Platt Scaling 将 SVM 的决策值映射为概率
svm_clf = CalibratedClassifierCV(linear_svc, method="sigmoid", cv=3)
svm_clf.fit(X_train_smote, y_train_smote)

y_pred_svm = svm_clf.predict(X_test)
y_prob_svm = svm_clf.predict_proba(X_test)[:, 1]

print("=== SVM（线性核 + SMOTE）模型评估 ===")
print(classification_report(y_test, y_pred_svm, target_names=["正常", "欺诈"]))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob_svm):.4f}")
print(f"PR-AUC:  {average_precision_score(y_test, y_prob_svm):.4f}")

---

## 6. 构建随机森林模型

In [ ]:
# Ref: https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html
rf_clf = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    max_features="sqrt",
    random_state=42,
    class_weight="balanced",  # 使用类别权重代替 SMOTE，两种策略均可
    n_jobs=-1,
)
# 直接使用原始训练集 + class_weight="balanced" 作为对比
rf_clf.fit(X_train, y_train)

y_pred_rf = rf_clf.predict(X_test)
y_prob_rf = rf_clf.predict_proba(X_test)[:, 1]

print("=== 随机森林（class_weight=balanced）模型评估 ===")
print(classification_report(y_test, y_pred_rf, target_names=["正常", "欺诈"]))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob_rf):.4f}")
print(f"PR-AUC:  {average_precision_score(y_test, y_prob_rf):.4f}")

---

## 7. 构建 XGBoost 模型

In [ ]:
# 计算正负样本比例
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scale_ratio = neg_count / pos_count
print(f"负类：{neg_count}, 正类：{pos_count}, 比例：{scale_ratio:.1f}")

# Ref: https://xgboost.readthedocs.io/en/stable/parameter.html
xgb_clf = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_ratio,  # 处理极端不平衡
    eval_metric="aucpr",           # 使用 PR-AUC 作为评估指标
    random_state=42,
    use_label_encoder=False,
    n_jobs=-1,
)
xgb_clf.fit(X_train, y_train)

y_pred_xgb = xgb_clf.predict(X_test)
y_prob_xgb = xgb_clf.predict_proba(X_test)[:, 1]

print("=== XGBoost（scale_pos_weight）模型评估 ===")
print(classification_report(y_test, y_pred_xgb, target_names=["正常", "欺诈"]))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob_xgb):.4f}")
print(f"PR-AUC:  {average_precision_score(y_test, y_prob_xgb):.4f}")

---

## 8. 构建 Isolation Forest（异常检测基线）

### 8.1 训练 Isolation Forest

Isolation Forest 是无监督方法，不使用标签进行训练。我们将其作为**基线方法**，观察不使用标签信息时模型能达到怎样的效果。

In [ ]:
# Ref: https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.IsolationForest.html
iso_clf = IsolationForest(
    n_estimators=200,
    contamination=0.00173,  # 告诉模型数据中异常比例约为 0.173%
    random_state=42,
    n_jobs=-1,
)
iso_clf.fit(X_train)  # 无监督，不使用 y_train

# Isolation Forest 输出 1（正常）和 -1（异常）
iso_pred_raw = iso_clf.predict(X_test)
# 转换为 0/1 标签：-1（异常）-> 1（欺诈），1（正常）-> 0
y_pred_iso = (iso_pred_raw == -1).astype(int)

# 使用 decision_function 的负值作为异常分数（越大越可能是异常）
iso_scores = -iso_clf.decision_function(X_test)

print("=== Isolation Forest 模型评估 ===")
print(classification_report(y_test, y_pred_iso, target_names=["正常", "欺诈"]))
print(f"ROC-AUC: {roc_auc_score(y_test, iso_scores):.4f}")
print(f"PR-AUC:  {average_precision_score(y_test, iso_scores):.4f}")

---

## 9. 模型对比与评估

### 9.1 PR 曲线对比（核心图表）

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- PR 曲线 ---
ax = axes[0]
for name, y_score in [
    ("SVM (线性核)", y_prob_svm),
    ("随机森林", y_prob_rf),
    ("XGBoost", y_prob_xgb),
    ("Isolation Forest", iso_scores),
]:
    precision_vals, recall_vals, _ = precision_recall_curve(y_test, y_score)
    ap = average_precision_score(y_test, y_score)
    ax.plot(recall_vals, precision_vals, label=f"{name} (AP={ap:.4f})")

# 基线：随机分类器的 PR-AUC = 正类比例
baseline = y_test.mean()
ax.axhline(y=baseline, color="gray", linestyle="--", label=f"随机基线 ({baseline:.4f})")
ax.set_xlabel("Recall（召回率）")
ax.set_ylabel("Precision（精确率）")
ax.set_title("PR 曲线对比（不平衡场景核心指标）")
ax.legend(loc="upper right")
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.05])

# --- ROC 曲线 ---
ax = axes[1]
for name, y_score in [
    ("SVM (线性核)", y_prob_svm),
    ("随机森林", y_prob_rf),
    ("XGBoost", y_prob_xgb),
    ("Isolation Forest", iso_scores),
]:
    fpr, tpr, _ = roc_curve(y_test, y_score)
    auc_val = roc_auc_score(y_test, y_score)
    ax.plot(fpr, tpr, label=f"{name} (AUC={auc_val:.4f})")

ax.plot([0, 1], [0, 1], "k--", label="随机猜测 (AUC=0.5)")
ax.set_xlabel("假正率 (FPR)")
ax.set_ylabel("真正率 (TPR / Recall)")
ax.set_title("ROC 曲线对比")
ax.legend(loc="lower right")

plt.tight_layout()
plt.show()

> **观察重点**：注意 ROC-AUC 的数值是否都很高（>0.95）。如果是，说明 ROC-AUC 在此场景下确实过于乐观。而 PR-AUC 能更真实地区分各模型的优劣。

### 9.2 评估指标汇总

In [ ]:
results = []
for name, y_pred, y_score in [
    ("SVM (线性核)", y_pred_svm, y_prob_svm),
    ("随机森林", y_pred_rf, y_prob_rf),
    ("XGBoost", y_pred_xgb, y_prob_xgb),
    ("Isolation Forest", y_pred_iso, iso_scores),
]:
    results.append({
        "模型": name,
        "Accuracy": round(accuracy_score(y_test, y_pred), 5),
        "Precision": round(precision_score(y_test, y_pred, zero_division=0), 4),
        "Recall": round(recall_score(y_test, y_pred), 4),
        "F1 Score": round(f1_score(y_test, y_pred), 4),
        "ROC-AUC": round(roc_auc_score(y_test, y_score), 4),
        "PR-AUC": round(average_precision_score(y_test, y_score), 4),
    })

results_df = pd.DataFrame(results)
print("\n=== 四模型评估指标汇总 ===")
display(results_df)

> **解读提示**：
> - 所有模型的 Accuracy 都在 99% 以上，但这 **毫无意义**——关注 PR-AUC 和 Recall。
> - Isolation Forest 作为无监督方法，Recall 通常较低，但不需要标签数据是其独特优势。
> - 有监督方法（RF、XGBoost）通常在 PR-AUC 上显著优于无监督方法。

### 9.3 混淆矩阵对比

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(22, 4))

for ax, y_pred, name in [
    (axes[0], y_pred_svm, "SVM (线性核)"),
    (axes[1], y_pred_rf, "随机森林"),
    (axes[2], y_pred_xgb, "XGBoost"),
    (axes[3], y_pred_iso, "Isolation Forest"),
]:
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["正常", "欺诈"], yticklabels=["正常", "欺诈"])
    ax.set_title(f"{name}\n混淆矩阵")
    ax.set_xlabel("预测类别")
    ax.set_ylabel("真实类别")

plt.tight_layout()
plt.show()

> **阅读混淆矩阵**：在反欺诈场景中，右下角的 TP（成功识别欺诈）和左下角的 FN（漏掉的欺诈）是核心关注点。**FN 越少越好**——每一笔漏掉的欺诈都意味着真实的经济损失。

---

## 10. 阈值分析与优化

### 10.1 为什么需要调整阈值？

默认分类阈值为 0.5，但在反欺诈场景中，这往往不是最优选择。降低阈值可以提高 Recall（捕获更多欺诈），但会降低 Precision（增加误报）。业务上需要在两者之间找到平衡。

### 10.2 阈值–指标关系图

选择 PR-AUC 最高的模型进行阈值分析：

In [ ]:
# 选取表现最好的模型进行阈值分析（以 XGBoost 为例，可根据实际结果调整）
best_model_name = "XGBoost"
best_y_prob = y_prob_xgb

precisions, recalls, thresholds_pr = precision_recall_curve(y_test, best_y_prob)
# precision_recall_curve 的输出中 precisions 和 recalls 比 thresholds 多一个元素
thresholds_pr = np.append(thresholds_pr, 1.0)

# 计算每个阈值对应的 F1
f1_scores = 2 * precisions * recalls / (precisions + recalls + 1e-8)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# --- Precision / Recall vs 阈值 ---
ax = axes[0]
ax.plot(thresholds_pr, precisions, label="Precision", color="steelblue")
ax.plot(thresholds_pr, recalls, label="Recall", color="crimson")
ax.plot(thresholds_pr, f1_scores, label="F1 Score", color="green", linestyle="--")
ax.set_xlabel("分类阈值")
ax.set_ylabel("指标值")
ax.set_title(f"{best_model_name}：阈值 vs Precision / Recall / F1")
ax.legend()
ax.set_xlim([0, 1])
ax.axvline(x=0.5, color="gray", linestyle=":", label="默认阈值 0.5")

# --- 找到最优 F1 阈值和 Recall≥0.95 阈值 ---
best_f1_idx = np.argmax(f1_scores)
best_f1_threshold = thresholds_pr[best_f1_idx]

# 找到 Recall >= 0.95 的最大阈值
recall_mask = recalls >= 0.95
if recall_mask.any():
    recall95_idx = np.where(recall_mask)[0][0]  # 最高阈值处
    # 从满足条件的索引中找最大阈值
    valid_thresholds = thresholds_pr[recall_mask]
    recall95_threshold = valid_thresholds.max()
else:
    recall95_threshold = 0.0

ax.axvline(x=best_f1_threshold, color="green", linestyle="--", alpha=0.7)
ax.text(best_f1_threshold + 0.02, 0.5, f"最优F1阈值={best_f1_threshold:.3f}", color="green")

print(f"最优 F1 阈值：{best_f1_threshold:.4f}，对应 F1={f1_scores[best_f1_idx]:.4f}")
print(f"Recall≥0.95 的阈值：{recall95_threshold:.4f}")

# --- PR 曲线标注关键点 ---
ax = axes[1]
ax.plot(recalls, precisions, color="steelblue")
ax.scatter(recalls[best_f1_idx], precisions[best_f1_idx],
           color="green", s=100, zorder=5, label=f"最优F1点 (阈值={best_f1_threshold:.3f})")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title(f"{best_model_name}：PR 曲线关键点标注")
ax.legend()

plt.tight_layout()
plt.show()

### 10.3 不同阈值下的模型表现

In [ ]:
threshold_candidates = [0.3, 0.4, 0.5, best_f1_threshold, recall95_threshold]
threshold_results = []

for t in sorted(set(threshold_candidates)):
    y_pred_t = (best_y_prob >= t).astype(int)
    threshold_results.append({
        "阈值": round(t, 4),
        "Precision": round(precision_score(y_test, y_pred_t, zero_division=0), 4),
        "Recall": round(recall_score(y_test, y_pred_t), 4),
        "F1": round(f1_score(y_test, y_pred_t), 4),
        "误报数 (FP)": int(((y_pred_t == 1) & (y_test == 0)).sum()),
        "漏报数 (FN)": int(((y_pred_t == 0) & (y_test == 1)).sum()),
    })

threshold_df = pd.DataFrame(threshold_results)
print(f"\n=== {best_model_name} 不同阈值下的表现 ===")
display(threshold_df)

> **阈值选择建议**：
> - **高安全场景**（如实时拦截大额交易）：选择低阈值使 Recall ≥ 0.95，接受较高误报率。被误报的交易可以通过人工复审确认。
> - **低干扰场景**（如用户通知）：选择最优 F1 阈值，在 Precision 和 Recall 之间取得平衡，避免过多打扰正常用户。

---

## 11. 特征重要性分析

虽然 V1–V28 已经过 PCA 脱敏，无法直接对应原始业务含义，但特征重要性分析仍能帮助我们理解**哪些主成分对欺诈识别贡献最大**。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 随机森林特征重要性
rf_importance = pd.Series(rf_clf.feature_importances_, index=X.columns)
top15_rf = rf_importance.sort_values(ascending=False).head(15)
sns.barplot(x=top15_rf.values, y=top15_rf.index, ax=axes[0], color="steelblue")
axes[0].set_title("随机森林 — Top 15 特征重要性")
axes[0].set_xlabel("重要性分数")

# XGBoost 特征重要性
xgb_importance = pd.Series(xgb_clf.feature_importances_, index=X.columns)
top15_xgb = xgb_importance.sort_values(ascending=False).head(15)
sns.barplot(x=top15_xgb.values, y=top15_xgb.index, ax=axes[1], color="darkorange")
axes[1].set_title("XGBoost — Top 15 特征重要性")
axes[1].set_xlabel("重要性分数")

plt.tight_layout()
plt.show()

> **思考**：观察两个模型的特征重要性排序是否一致。如果某些特征（如 V14、V17）在两个模型中都排名靠前，说明这些主成分方向上包含了区分欺诈与正常交易的关键信息。

---

## 12. 实验结论

### 12.1 不平衡数据处理的必要性

在欺诈比例仅为 0.173% 的数据上，不进行任何不平衡处理的模型会完全忽略少数类。本实验验证了 SMOTE 过采样和 `class_weight` / `scale_pos_weight` 参数调整的有效性。

### 12.2 四种模型的比较

| 模型 | 优势 | 劣势 | 适用场景 |
|------|------|------|----------|
| **SVM（线性核）** | 理论优雅，适合高维数据 | 大规模数据训练慢，需标准化，调参敏感 | 中小规模数据、线性可分场景 |
| **随机森林** | 鲁棒性强，支持并行，不易过拟合 | 对极端不平衡敏感，模型体积大 | 通用基线，特征重要性分析 |
| **XGBoost** | 性能最优，内置不平衡处理，支持正则化 | 调参复杂，可解释性不如单棵树 | 工业级反欺诈系统 |
| **Isolation Forest** | 无需标签，天然适合异常检测 | 精度通常不如有监督方法 | 冷启动阶段、无标签数据场景 |

### 12.3 评估指标的选择

- 在极端不平衡数据上，**Accuracy 不可信，PR-AUC 是核心指标**。
- ROC-AUC 仍有参考价值，但需结合 PR-AUC 一起分析。
- 实际业务中需根据风控策略选择分类阈值，在 Recall 与 Precision 之间做出明确取舍。

### 12.4 阈值选择的业务意义

阈值不是一个纯技术问题，而是一个**业务决策**：
- 阈值越低 → Recall 越高 → 漏掉的欺诈越少 → 但误报越多 → 用户体验下降、人工复审成本增加。
- 阈值越高 → Precision 越高 → 误报越少 → 但漏掉更多欺诈 → 经济损失增加。
- 最终阈值应由**风控团队、产品团队和技术团队共同决定**，基于误报成本与漏报损失的量化分析。

---

## 13. 课堂思考题

1. 如果将 SMOTE 应用到整个数据集（包括测试集）再划分训练/测试集，评估结果会怎样变化？为什么这种做法是错误的？
2. SVM 使用 RBF 核在此数据集上是否可行？主要瓶颈是什么？如果必须使用 RBF 核，有哪些加速策略？
3. Isolation Forest 的 `contamination` 参数设置为 0.00173 是基于先验知识。如果你不知道实际欺诈比例，应该如何设置？
4. 假设每笔漏检欺诈的平均损失为 5000 元，每笔误报的处理成本为 10 元，如何计算不同阈值下的总期望成本？哪个阈值能最小化总成本？
5. 在实际的反欺诈系统中，模型通常不是独立工作的，而是作为"规则引擎 + 模型"组合系统的一部分。你认为规则引擎和机器学习模型各自的优势是什么？如何结合使用？
